# Three-Class Groundwater Drought Recovery Groups

This notebook assigns three recovery groups across the MRVA domain using physically constrained response prototypes. The classification uses drought-response metrics only; pumping and resistivity are withheld for interpretation.

In [ ]:
from pathlib import Path


import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

ROOT = next(
    p for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    if (p / 'outputs' / 'RECON_MAIN_2011_2023').exists()
)
RECON = ROOT / 'outputs' / 'RECON_MAIN_2011_2023'
METRICS_DIR = RECON / 'metrics'
METRICS_CSV = METRICS_DIR / 'drought_metrics.csv'
OUT_DIR = METRICS_DIR / 'clustering'
FIG_DIR = OUT_DIR / 'figures'
for d in [OUT_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def display_path(path):
    try:
        return str(Path(path).resolve().relative_to(ROOT))
    except ValueError:
        return str(path)

REQUIRED_FEATURES = [
    'decline_m',
    'Rdown_m_per_month',
    'RR_early',
    'T50_months',
]
MEMORY_FEATURES = [
    'RR2019',
]
FEATURES = REQUIRED_FEATURES + MEMORY_FEATURES
CLASS_ORDER = ['Fast recovery', 'Slow recovery', 'Buffered']
CLASS_COLORS = {
    'Fast recovery': '#2D5FB8',
    'Slow recovery': '#C44E72',
    'Buffered': '#13A8A2',
}

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans', 'sans-serif'],
    'font.size': 7,
    'axes.spines.right': False,
    'axes.spines.top': False,
    'axes.linewidth': 0.7,
    'legend.frameon': False,
})

def save_figure(fig, stem, dpi=600):
    out = FIG_DIR / f'{stem}.png'
    fig.savefig(out, bbox_inches='tight', dpi=dpi)
    return [out]

print('Metrics:', display_path(METRICS_CSV))
print('Output:', display_path(OUT_DIR))

## 1. Response Class Definitions

| Three-class group | Hydrograph signature | Physical emphasis |
|---|---|---|
| `Fast recovery` | Clear drought decline followed by rapid early recovery | Responsive but resilient water-level dynamics |
| `Slow recovery` | Delayed, incomplete, or persistent recovery after drought | Long memory expressed by low early recovery, long half-recovery time, or elevated 2019 deficit |
| `Buffered` | Small decline and low long-term deficit | Weak drought response and comparatively stable water levels |



In [ ]:
df = pd.read_csv(METRICS_CSV)
missing = [c for c in ['grid_id', 'row', 'col', 'x', 'y'] + FEATURES if c not in df.columns]
if missing:
    raise KeyError(f'Missing required metric columns: {missing}')

valid_for_clustering = df[REQUIRED_FEATURES].notna().all(axis=1)
valid_df = df.loc[valid_for_clustering].copy()
print(f'Total MRVA grid cells: {len(df):,}')
print(f'Valid cells for classification: {len(valid_df):,} ({len(valid_df) / len(df) * 100:.2f}%)')


## 2. Convert Metrics To Response Coordinates

Each metric is converted to a percentile-rank coordinate from 0 to 1. This avoids clipping raw physical values while keeping the class scoring robust to skewed recovery fractions.


In [ ]:
def percentile_rank_01(values):
    return pd.Series(values).rank(pct=True, method='average').to_numpy(dtype=float)

rank_inputs = pd.DataFrame(index=valid_df.index)
rank_inputs['decline_for_rank'] = valid_df['decline_m']
rank_inputs['Rdown_for_rank'] = valid_df['Rdown_m_per_month']
rank_inputs['RR_early_for_rank'] = valid_df['RR_early'].clip(lower=0.0, upper=2.0)
rank_inputs['T50_for_rank'] = valid_df['T50_months'].clip(lower=1.0, upper=72.0)
rank_inputs['RR2019_for_rank'] = valid_df['RR2019'].clip(lower=-1.0, upper=5.0)

rank_df = pd.DataFrame(index=valid_df.index)
rank_df['decline_rank'] = percentile_rank_01(rank_inputs['decline_for_rank'])
rank_df['Rdown_rank'] = percentile_rank_01(rank_inputs['Rdown_for_rank'])
rank_df['RR_early_rank'] = percentile_rank_01(rank_inputs['RR_early_for_rank'])
rank_df['T50_rank'] = percentile_rank_01(rank_inputs['T50_for_rank'])
rank_df['RR2019_rank'] = percentile_rank_01(rank_inputs['RR2019_for_rank'])
rank_df['RR2019_deficit_rank'] = 1.0 - rank_df['RR2019_rank']
rank_df['RR2019_deficit_rank'] = rank_df['RR2019_deficit_rank'].fillna(0.5)


## 3. Assign Classes

The classifier assigns the three recovery groups directly using weighted L1 distance to physically defined response prototypes in percentile-rank space. Classification ranks use capped recovery variables (`T50` capped at 72 months and `RR2019` capped from -1 to 5) to reduce leverage from long tails and small-deficit ratios. `Buffered` also has a strict weak-response, low-deficit gate so cells with low decline, low decline rate and low 2019 deficit are not pulled into slow-recovery classes by noisy recovery ratios.


In [ ]:
rank_features = ['decline_rank', 'Rdown_rank', 'RR_early_rank', 'T50_rank', 'RR2019_deficit_rank']
X = rank_df[rank_features].to_numpy(dtype=float)

class_targets = pd.DataFrame(
    {
        'decline_rank': [0.78, 0.62, 0.10],
        'Rdown_rank': [0.65, 0.55, 0.12],
        'RR_early_rank': [0.88, 0.25, 0.60],
        'T50_rank': [0.12, 0.88, 0.28],
        'RR2019_deficit_rank': [0.35, 0.80, 0.12],
    },
    index=CLASS_ORDER,
)
class_weights = pd.DataFrame(
    {
        'decline_rank': [0.20, 0.15, 0.35],
        'Rdown_rank': [0.16, 0.10, 0.30],
        'RR_early_rank': [0.32, 0.25, 0.08],
        'T50_rank': [0.24, 0.35, 0.07],
        'RR2019_deficit_rank': [0.08, 0.15, 0.20],
    },
    index=CLASS_ORDER,
)
class_weights = class_weights.div(class_weights.sum(axis=1), axis=0)

class_distances = np.zeros((len(valid_df), len(CLASS_ORDER)), dtype=float)
for j, class_name in enumerate(CLASS_ORDER):
    target = class_targets.loc[class_name, rank_features].to_numpy(dtype=float)
    weight = class_weights.loc[class_name, rank_features].to_numpy(dtype=float)
    class_distances[:, j] = (weight * np.abs(X - target)).sum(axis=1)

class_order_idx = np.argsort(class_distances, axis=1)
best_idx = class_order_idx[:, 0].copy()
assignment_rule = np.full(len(valid_df), 'nearest_three_class_prototype', dtype=object)

buffered_idx = CLASS_ORDER.index('Buffered')
buffered_gate = ((rank_df['decline_rank'] <= 0.22) & (rank_df['Rdown_rank'] <= 0.28) & (rank_df['RR2019_deficit_rank'] <= 0.40)).to_numpy(dtype=bool)
best_idx[buffered_gate] = buffered_idx
assignment_rule[buffered_gate] = 'buffered_weak_response_low_deficit_gate'
second_idx = np.array([
    next(idx for idx in ranked if idx != selected)
    for ranked, selected in zip(class_order_idx, best_idx)
], dtype=int)

best_class = np.array(CLASS_ORDER)[best_idx]
second_class = np.array(CLASS_ORDER)[second_idx]
best_distance = class_distances[np.arange(len(valid_df)), best_idx]
second_distance = class_distances[np.arange(len(valid_df)), second_idx]
best_score = 1.0 - best_distance
second_score = 1.0 - second_distance

valid_df['response_class'] = best_class
valid_df['best_score'] = best_score
valid_df['second_score'] = second_score
valid_df['score_margin'] = best_score - second_score
valid_df['second_class'] = second_class
valid_df['assignment_rule'] = assignment_rule

labels_df = df[['grid_id', 'row', 'col', 'x', 'y'] + FEATURES].copy()
labels_df['valid_for_clustering'] = valid_for_clustering
labels_df['response_class'] = pd.Series(pd.NA, index=labels_df.index, dtype='object')
labels_df['best_score'] = np.nan
labels_df['second_class'] = pd.Series(pd.NA, index=labels_df.index, dtype='object')
labels_df['second_score'] = np.nan
labels_df['score_margin'] = np.nan
labels_df['assignment_rule'] = pd.Series(pd.NA, index=labels_df.index, dtype='object')
for col in rank_df.columns:
    labels_df[col] = np.nan

labels_df.loc[valid_df.index, 'response_class'] = valid_df['response_class'].to_numpy()
labels_df.loc[valid_df.index, 'best_score'] = valid_df['best_score'].to_numpy()
labels_df.loc[valid_df.index, 'second_class'] = valid_df['second_class'].to_numpy()
labels_df.loc[valid_df.index, 'second_score'] = valid_df['second_score'].to_numpy()
labels_df.loc[valid_df.index, 'score_margin'] = valid_df['score_margin'].to_numpy()
labels_df.loc[valid_df.index, 'assignment_rule'] = valid_df['assignment_rule'].to_numpy()
for col in rank_df.columns:
    labels_df.loc[valid_df.index, col] = rank_df[col].to_numpy()

labels_df.to_csv(OUT_DIR / 'cluster_labels.csv', index=False)

counts = valid_df['response_class'].value_counts().reindex(CLASS_ORDER).rename('n').reset_index()
counts = counts.rename(columns={'index': 'response_class'})
counts['area_fraction'] = counts['n'] / len(valid_df)
print('Three-class response groups:')
print(counts.to_string(index=False))


## 4. Plot And Save Clustering Results

The map below is the only figure produced by this notebook: the three-class recovery-group map.


In [ ]:
plot_df = labels_df[labels_df['valid_for_clustering']].copy()
plot_df['class_id'] = plot_df['response_class'].map({a: i for i, a in enumerate(CLASS_ORDER)})
rows_arr = labels_df['row'].to_numpy(dtype=int)
cols_arr = labels_df['col'].to_numpy(dtype=int)
n_rows = int(np.nanmax(rows_arr)) + 1
n_cols = int(np.nanmax(cols_arr)) + 1
origin_mode = 'upper' if labels_df[['row', 'y']].corr().loc['row', 'y'] < 0 else 'lower'

class_grid = np.full((n_rows, n_cols), np.nan, dtype=float)
class_grid[plot_df['row'].to_numpy(dtype=int), plot_df['col'].to_numpy(dtype=int)] = plot_df['class_id'].to_numpy(dtype=float)

cmap = ListedColormap([CLASS_COLORS[a] for a in CLASS_ORDER])
cmap.set_bad('#FFFFFF')
norm = BoundaryNorm(np.arange(-0.5, len(CLASS_ORDER) + 0.5, 1), cmap.N)
class_counts = plot_df['response_class'].value_counts().reindex(CLASS_ORDER)

fig, ax = plt.subplots(figsize=(4.7, 6.4))
ax.imshow(class_grid, cmap=cmap, norm=norm, origin=origin_mode, interpolation='nearest')
ax.set_title('Groundwater drought recovery groups', fontsize=8, pad=4)
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)
legend_handles = [
    mpl.patches.Patch(
        facecolor=CLASS_COLORS[class_name],
        edgecolor='#1F1F1F',
        linewidth=0.35,
        label=f"{class_name} ({class_counts.loc[class_name] / len(plot_df) * 100:.1f}%)",
    )
    for class_name in CLASS_ORDER
]
ax.legend(
    handles=legend_handles,
    loc='center left',
    bbox_to_anchor=(1.01, 0.50),
    title='Group',
    title_fontsize=6.5,
    fontsize=6,
    handlelength=1.0,
    handleheight=0.8,
    borderaxespad=0.0,
)
fig.tight_layout()
save_figure(fig, 'cluster_map')
plt.show()


In [ ]:
print('Saved clustering outputs:')
for path in [
    OUT_DIR / 'cluster_labels.csv',
    FIG_DIR / 'cluster_map.png',
]:
    print(' ', display_path(path))
